## load dataset

In [1]:
import json
import math



# Load Dataset

with open("text_segmentation_dataset.json", "r") as f:
    data = json.load(f)

word_counts = data["word_counts"]
test_cases = data["test_cases"]

TOTAL_WORDS = data["metadata"]["total_corpus_words"]

vocabulary = set(word_counts.keys())
MAX_WORD_LENGTH = max(len(word) for word in vocabulary)

## evaluation

In [2]:
# Evaluation: Levenshtein Edit Distance

def edit_distance(a, b):
    """
    Calculate Levenshtein edit distance between two strings.
    """

    m = len(a)
    n = len(b)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if a[i - 1] == b[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,        # deletion
                dp[i][j - 1] + 1,        # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    return dp[m][n]

## greedy longest matching

In [3]:
# 1. GREEDY LONGEST MATCHING

def greedy_segmentation(text):

    result = []
    i = 0

    while i < len(text):

        longest_word = None

        # Try longest possible word first
        max_len = min(MAX_WORD_LENGTH, len(text) - i)

        for length in range(max_len, 0, -1):

            candidate = text[i:i + length]

            if candidate in vocabulary:
                longest_word = candidate
                break

        # Safety fallback
        if longest_word is None:
            longest_word = text[i]

        result.append(longest_word)

        i += len(longest_word)

    return result

## dynamic programming

In [4]:
# 2. DYNAMIC PROGRAMMING

def dp_segmentation(text):

    n = len(text)

    # dp[i] = maximum log probability for text[:i]
    dp = [-float("inf")] * (n + 1)

    # Previous position used to reconstruct solution
    previous = [None] * (n + 1)

    dp[0] = 0

    for i in range(n):

        if dp[i] == -float("inf"):
            continue

        max_len = min(MAX_WORD_LENGTH, n - i)

        for length in range(1, max_len + 1):

            word = text[i:i + length]

            if word in word_counts:

                # P(word) = frequency / total number of corpus words
                probability = word_counts[word] / TOTAL_WORDS

                log_probability = math.log(probability)

                new_score = dp[i] + log_probability

                if new_score > dp[i + length]:

                    dp[i + length] = new_score

                    previous[i + length] = (i, word)

    # Reconstruct segmentation
    result = []

    position = n

    # If DP could not segment completely
    if previous[position] is None:
        return greedy_segmentation(text)

    while position > 0:

        prev_position, word = previous[position]

        result.append(word)

        position = prev_position

    result.reverse()

    return result

In [5]:
# EVALUATION

def evaluate(method):

    exact_matches = 0

    correct_words = 0
    total_words = 0

    total_edit_distance = 0

    for test in test_cases:

        input_text = test["input"]

        # Ground truth words
        ground_truth = test["ground_truth"].split()

        # Prediction
        prediction = method(input_text)

       
        # Exact segmentation accuracy
      

        if prediction == ground_truth:
            exact_matches += 1

        
        # Word accuracy
      

        for predicted, actual in zip(prediction, ground_truth):

            if predicted == actual:
                correct_words += 1

        total_words += len(ground_truth)

       
        # Edit distance
        

        predicted_text = " ".join(prediction)
        actual_text = " ".join(ground_truth)

        total_edit_distance += edit_distance(
            predicted_text,
            actual_text
        )

    # Metrics
    exact_accuracy = exact_matches / len(test_cases)

    word_accuracy = correct_words / total_words

    average_edit_distance = (
        total_edit_distance / len(test_cases)
    )

    return exact_accuracy, word_accuracy, average_edit_distance



# Run Both Algorithms


greedy_results = evaluate(greedy_segmentation)

dp_results = evaluate(dp_segmentation)

In [6]:
# Print Results


print("=" * 60)
print("TEXT SEGMENTATION RESULTS")
print("=" * 60)

print("\nGreedy Based Approach")
print("-" * 40)

print(
    f"Exact Sentence Accuracy : "
    f"{greedy_results[0] * 100:.2f}%"
)

print(
    f"Word Accuracy           : "
    f"{greedy_results[1] * 100:.2f}%"
)

print(
    f"Average Edit Distance   : "
    f"{greedy_results[2]:.3f}"
)


print("\nDynamic Programming Approach")
print("-" * 40)

print(
    f"Exact Sentence Accuracy : "
    f"{dp_results[0] * 100:.2f}%"
)

print(
    f"Word Accuracy           : "
    f"{dp_results[1] * 100:.2f}%"
)

print(
    f"Average Edit Distance   : "
    f"{dp_results[2]:.3f}"
)

TEXT SEGMENTATION RESULTS

Greedy Based Approach
----------------------------------------
Exact Sentence Accuracy : 69.10%
Word Accuracy           : 82.21%
Average Edit Distance   : 1.290

Dynamic Programming Approach
----------------------------------------
Exact Sentence Accuracy : 98.20%
Word Accuracy           : 99.14%
Average Edit Distance   : 0.028


## sample examples

In [7]:
# Sample Examples

sample_tests = test_cases[:2]  # pick first two test cases

for test in sample_tests:

    input_text = test["input"]

    greedy_result = greedy_segmentation(input_text)
    dp_result = dp_segmentation(input_text)

    print("Given statement:")
    print(input_text)

    print("\nGreedy approach:")
    print(" ".join(greedy_result))

    print("\nDynamic Programming approach:")
    print(" ".join(dp_result))

    print("\n" + "=" * 60 + "\n")

Given statement:
itthatthecitytakestepstothisproblem

Greedy approach:
it that the city takes t e p s to this problem

Dynamic Programming approach:
it that the city take steps to this problem


Given statement:
oftitlelawwasalsobythe

Greedy approach:
of title law was also by the

Dynamic Programming approach:
of title law was also by the


